<a href="https://colab.research.google.com/github/Sanket456/VoltRelay-Energy-Data-Analytics/blob/main/Hackathalon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os


drive.mount('/content/drive')

project_folder = '/content/drive/MyDrive/Hackathalon'

if not os.path.exists(project_folder):
    os.makedirs(project_folder)
    print(f"Created new folder: {project_folder}")

os.chdir(project_folder)
print(f"Current Working Directory: {os.getcwd()}")

print("\nFiles in folder:")
print(os.listdir('.'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current Working Directory: /content/drive/MyDrive/Hackathalon

Files in folder:
['city_daily_context.csv', 'fleet_partners.csv', 'stations.csv', 'batteries.csv', 'riders.csv', 'swap_events.csv', 'support_tickets.csv', 'station_hourly_status.csv', 'city_daily_context_transformed.csv', 'batteries_transformed.csv', 'riders_transformed.csv', 'fleet_partners_transformed.csv', 'station_hourly_status_transformed.csv', 'support_tickets_transformed.csv', 'swap_events_transformed.parquet', 'stations_transformed.csv', 'swap_events_transformed.csv']


In [ ]:
import pandas as pd
import numpy as np

def transform_city_daily_context(file_path='city_daily_context.csv'):

    df = pd.read_csv(file_path)


    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(by=['city', 'date']).reset_index(drop=True)


    df['festival_or_event'] = df['festival_or_event'].fillna('None')


    df['heat_alert'] = (df['max_temp_c'] >= 42.0).astype(int)


    rain_bins = [-0.1, 0.0, 15.0, 50.0, 100.0, float('inf')]
    rain_labels = ['None', 'Light', 'Moderate', 'Heavy', 'Severe']
    df['rainfall_tier'] = pd.cut(df['rainfall_mm'], bins=rain_bins, labels=rain_labels)


    df.loc[df['rainfall_mm'] >= 100.0, 'flood_disruption'] = True


    return df


df_city_clean = transform_city_daily_context('city_daily_context.csv')
df_city_clean.to_csv('city_daily_context_transformed.csv', index=False)

print("Transformation Complete! Saved 'city_daily_context_transformed.csv' to Drive.")
df_city_clean.head()

Transformation Complete! Saved 'city_daily_context_transformed.csv' to Drive.


,city,date,max_temp_c,min_temp_c,rainfall_mm,heat_alert,flood_disruption,festival_or_event,is_public_holiday,grid_outage_hours,competitor_promo_active,ecommerce_sale_event,rainfall_tier
0,Bengaluru,2024-01-01,16.7,6.7,0.0,0,False,None,False,0.0,False,False,None
1,Bengaluru,2024-01-02,17.3,4.6,1.8,0,False,None,False,0.2,False,False,Light
2,Bengaluru,2024-01-03,13.9,6.9,0.7,0,False,None,False,0.0,False,False,Light
3,Bengaluru,2024-01-04,17.9,4.0,2.2,0,False,None,False,0.1,False,False,Light
4,Bengaluru,2024-01-05,16.0,7.8,2.0,0,False,None,False,0.0,False,False,Light


In [ ]:
import numpy as np
import pandas as pd

def transform_fleet_partners(file_path='fleet_partners.csv', output_filepath='fleet_partners_transformed.csv'):
    df = pd.read_csv(file_path)


    date_cols = ['contract_start_date', 'amendment_date']
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])


    df['is_amended'] = df['amendment_date'].notnull().astype(int)
    df['is_peak_surcharge_billable'] = (df['peak_surcharge_billable'].astype(str).str.upper() == 'Y').astype(int)
    df['num_cities_active'] = df['cities_active'].apply(lambda x: len(str(x).split('|')) if pd.notnull(x) else 0)

    if output_filepath:
        df.to_csv(output_filepath, index=False)
        print(f"Transformation complete! Saved '{output_filepath}'.")
        print(f"Dataset shape: {df.shape}")
        print("\nAmended Contracts:")
        display_cols = ['partner_id', 'partner_name', 'discount_pct', 'discount_pct_after_amendment', 'amendment_date']
        print(df[df['is_amended'] == 1][[c for c in display_cols if c in df.columns]])

    return df

df_fleet_clean = transform_fleet_partners('fleet_partners.csv')

Transformation complete! Saved 'fleet_partners_transformed.csv'.
Dataset shape: (12, 15)

Amended Contracts:
  partner_id partner_name  discount_pct  discount_pct_after_amendment  \
2      FP-03      ZipDrop          12.0                          28.0   

  amendment_date  
2     2024-11-01  


In [ ]:
import numpy as np
import pandas as pd

def transform_stations(file_path='stations.csv', output_filepath='stations_transformed.csv'):
    df = pd.read_csv(file_path)


    date_cols = ['commissioned_date', 'decommissioned_date', 'firmware_updated_date', 'competitor_within_1_5km_since']
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])


    df['is_test_station'] = df['station_id'].str.startswith('STN-TST').astype(int)
    df['is_decommissioned'] = df['decommissioned_date'].notnull().astype(int)
    df['total_slots'] = df['slots_2w'] + df['slots_3w']
    df['total_inventory_target'] = df['inventory_target_2w'] + df['inventory_target_3w']
    df['monthly_fixed_cost_inr'] = df['monthly_rent_inr'] + df['monthly_maintenance_inr']
    df['has_competitor_nearby'] = df['competitor_within_1_5km_since'].notnull().astype(int)

    if output_filepath:
        df.to_csv(output_filepath, index=False)
        print(f"Transformation complete! Saved '{output_filepath}'.")
        print(f"Dataset shape: {df.shape}")
        print(f"Total test stations tagged: {df['is_test_station'].sum()}")
        print("\nStations by City:")
        print(df['city'].value_counts())

    return df

df_stations_clean = transform_stations('stations.csv')

Transformation complete! Saved 'stations_transformed.csv'.
Dataset shape: (152, 28)
Total test stations tagged: 2

Stations by City:
city
Bengaluru    34
Delhi NCR    30
Hyderabad    26
Pune         22
Mumbai       22
Jaipur       18
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np

def transform_batteries(file_path='batteries.csv', output_filepath='batteries_transformed.csv', snapshot_date='2025-06-30'):
    df = pd.read_csv(file_path)


    date_cols = ['manufacture_date', 'commission_date', 'retired_date']
    for col in date_cols:
        df[col] = pd.to_datetime(df[col])


    df['is_retired'] = df['retired_date'].notnull().astype(int)
    df['retirement_reason'] = df['retirement_reason'].fillna('active')


    df['commission_delay_days'] = (df['commission_date'] - df['manufacture_date']).dt.days

    end_ref = pd.to_datetime(snapshot_date)
    df['effective_end_date'] = df['retired_date'].fillna(end_ref)
    df['days_in_service'] = (df['effective_end_date'] - df['commission_date']).dt.days
    df.drop(columns=['effective_end_date'], inplace=True)


    df['soh_degradation_pct'] = (df['initial_soh_pct'] - df['current_soh_pct']).round(2)
    df['daily_degradation_rate'] = (df['soh_degradation_pct'] / np.maximum(df['days_in_service'], 1)).round(4)


    df['cost_per_kwh'] = (df['purchase_cost_inr'] / df['rated_capacity_kwh']).round(2)
    df['vehicle_class'] = df['pack_type'].apply(lambda x: '2W' if '2W' in x else '3W')


    if output_filepath:
        df.to_csv(output_filepath, index=False)
        print(f"Transformation complete! Cleaned data saved successfully to '{output_filepath}'.")
        print(f"New dataset shape: {df.shape}")

    return df


df_batteries_clean = transform_batteries('batteries.csv')
df_batteries_clean.head(3)

Transformation complete! Cleaned data saved successfully to 'batteries_transformed.csv'.
New dataset shape: (6500, 20)


,battery_id,pack_type,supplier,manufacturing_lot,manufacture_date,commission_date,rated_capacity_kwh,purchase_cost_inr,initial_soh_pct,bms_firmware,retired_date,retirement_reason,current_soh_pct,is_retired,commission_delay_days,days_in_service,soh_degradation_pct,daily_degradation_rate,cost_per_kwh,vehicle_class
0,BAT-2W-100000,2W_2.1kWh,Cellora,CE-2503,2025-01-30,2025-03-19,2.1,31018,99.9,BMS-4.1,NaT,active,79.5,0,48,103,20.4,0.1981,14770.48,2W
1,BAT-2W-100001,2W_2.1kWh,Cellora,CE-2403,2024-02-27,2024-03-30,2.1,34111,99.2,BMS-4.2,NaT,active,79.5,0,32,457,19.7,0.0431,16243.33,2W
2,BAT-3W-100002,3W_4.8kWh,Cellora,CE-2305,2023-04-15,2023-05-11,4.8,66521,99.5,BMS-4.1,NaT,active,87.0,0,26,781,12.5,0.0160,13858.54,3W


In [ ]:
import pandas as pd
import numpy as np

def transform_riders(file_path='riders.csv', snapshot_date='2025-06-30'):

    df = pd.read_csv(file_path)


    df['signup_date'] = pd.to_datetime(df['signup_date'])
    end_ref = pd.to_datetime(snapshot_date)


    df['account_age_days'] = (end_ref - df['signup_date']).dt.days


    cutoff_60d = end_ref - pd.Timedelta(days=60)
    df['is_observable_60d'] = (df['signup_date'] <= cutoff_60d).astype(int)


    city_map = {
        'bengaluru': 'Bengaluru',
        'blr': 'Bengaluru',
        'bangalore': 'Bengaluru',
        'delhi ncr': 'Delhi NCR',
        'delhi': 'Delhi NCR',
        'new delhi': 'Delhi NCR',
        'gurgaon': 'Delhi NCR',
        'hyderabad': 'Hyderabad',
        'hyd': 'Hyderabad',
        'pune': 'Pune',
        'pun': 'Pune',
        'mumbai': 'Mumbai',
        'bombay': 'Mumbai',
        'mum': 'Mumbai',
        'jaipur': 'Jaipur',
        'jai': 'Jaipur'
    }

    df['home_city_raw'] = df['home_city']
    df['home_city'] = df['home_city'].astype(str).str.strip().str.lower().map(city_map)


    assert df['home_city'].isnull().sum() == 0, f"Unmapped city values detected: {df[df['home_city'].isnull()]['home_city_raw'].unique()}"
    df.drop(columns=['home_city_raw'], inplace=True)



    return df


df_riders_clean = transform_riders('riders.csv')
df_riders_clean.to_csv('riders_transformed.csv', index=False)
print("Transformation Complete! Saved 'riders_transformed.csv' to Drive.")
df_riders_clean.head()

Transformation Complete! Saved 'riders_transformed.csv' to Drive.


,rider_id,partner_id,vehicle_class,vehicle_model,home_zone,signup_date,signup_channel,plan_type,declared_shift,age_band,kyc_verified,home_city,account_age_days,is_observable_60d
0,RD-1000000,FP-04,3W,Kranti Cargo3,DEL-Noida,2023-03-08,field_agent,partner_billed,night,45+,True,Delhi NCR,845,1
1,RD-1000001,FP-05,2W,Volt Cub,MUM-Thane,2023-03-08,partner_onboarding,partner_billed,mixed,35-44,True,Mumbai,845,1
2,RD-1000002,FP-06,2W,Torq X1,HYD-West,2023-03-08,partner_onboarding,partner_billed,evening,25-34,False,Hyderabad,845,1
3,RD-1000003,FP-01,2W,Zipp E2,MUM-Navi Mumbai,2023-03-08,partner_onboarding,partner_billed,evening,25-34,True,Mumbai,845,1
4,RD-1000004,FP-02,2W,Zipp E2,HYD-Hitech City,2023-03-08,partner_onboarding,partner_billed,night,NaN,True,Hyderabad,845,1


In [ ]:
import numpy as np
import pandas as pd

def transform_support_tickets(file_path='support_tickets.csv', output_filepath='support_tickets_transformed.csv'):
    df = pd.read_csv(file_path)


    df['created_ts'] = pd.to_datetime(df['created_ts'])



    df['is_resolved'] = (df['resolution_status'].astype(str).str.lower() == 'resolved').astype(int)


    df['has_csat'] = df['csat_score'].notnull().astype(int)


    if output_filepath:
        df.to_csv(output_filepath, index=False)
        print(f"Transformation complete! Saved '{output_filepath}'.")
        print(f"Dataset shape: {df.shape}")
        print("\nTicket Categories:")
        print(df['category'].value_counts())

    return df

df_tickets_clean = transform_support_tickets('support_tickets.csv')

Transformation complete! Saved 'support_tickets_transformed.csv'.
Dataset shape: (44000, 14)

Ticket Categories:
category
no_battery_available    11647
other                    9484
low_range                6636
app_issue                5787
long_queue               5731
billing_dispute          4715
Name: count, dtype: int64


In [ ]:
import numpy as np
import pandas as pd

def transform_station_hourly_status(file_path='station_hourly_status.csv', output_filepath='station_hourly_status_transformed.csv'):
    print("Loading station_hourly_status.csv (1.49M rows)...")
    df = pd.read_csv(file_path)


    df['hour_start'] = pd.to_datetime(df['hour_start'])



    df['thermal_stress_flag'] = np.where(
        (df['cabinet_temp_c'] >= 45.0) | (df['ambient_temp_c'] >= 40.0), 1, 0
    )
    df['has_grid_outage'] = (df['outage_minutes'] > 0).astype(int)

    if output_filepath:
        print("Saving station_hourly_status_transformed.csv to Drive...")
        df.to_csv(output_filepath, index=False)
        print(f"Transformation complete! Saved '{output_filepath}'.")
        print(f"Dataset shape: {df.shape}")
        print("\nTelemetry Status Breakdown:")
        print(df['telemetry_status'].value_counts(dropna=False))

    return df

df_hourly_clean = transform_station_hourly_status('station_hourly_status.csv')

Loading station_hourly_status.csv (1.49M rows)...
Saving station_hourly_status_transformed.csv to Drive...
Transformation complete! Saved 'station_hourly_status_transformed.csv'.
Dataset shape: (1487712, 16)

Telemetry Status Breakdown:
telemetry_status
ok         1441341
partial      36987
missing       9384
Name: count, dtype: int64


In [ ]:
import gc
import numpy as np
import pandas as pd

def transform_swap_events(file_path='swap_events.csv', output_filepath='swap_events_transformed.csv'):
    print("Loading swap_events.csv (3.88M rows)...")
    df = pd.read_csv(file_path)

    # 1. Firmware v3.2.0 timestamp bug (+5h30m)
    df['event_ts'] = pd.to_datetime(df['event_ts'])
    bug_mask = (
        (df['station_firmware'] == 'v3.2.0') &
        (df['event_ts'] >= '2025-03-10') &
        (df['event_ts'] <= '2025-04-14 23:59:59')
    )
    df['is_firmware_bug_ts'] = bug_mask.astype(int)
    df['event_ts_corrected'] = df['event_ts']
    df.loc[bug_mask, 'event_ts_corrected'] = df.loc[bug_mask, 'event_ts'] + pd.Timedelta(hours=5, minutes=30)
    print(f"Firmware-corrected rows: {bug_mask.sum()}")


    print("Flagging offline sync retries...")
    df = df.sort_values(by=['rider_id', 'station_id', 'event_ts_corrected']).reset_index(drop=True)

    prev_rider = df['rider_id'].shift(1)
    prev_station = df['station_id'].shift(1)
    prev_ts = df['event_ts_corrected'].shift(1)
    time_diff_sec = (df['event_ts_corrected'] - prev_ts).dt.total_seconds()

    is_dup = (
        (df['rider_id'] == prev_rider) &
        (df['station_id'] == prev_station) &
        (df['sync_mode'] == 'offline_batch') &
        (time_diff_sec >= 0) & (time_diff_sec <= 300)
    )
    df['is_near_duplicate'] = is_dup.astype(int)

    del prev_rider, prev_station, prev_ts, time_diff_sec
    gc.collect()
    print(f"Near-duplicates flagged: {df['is_near_duplicate'].sum()}")


    df['km_outlier_flag'] = ((df['km_since_last_swap'] < 0) | (df['km_since_last_swap'] > 300)).astype(int)
    df['sensor_drift_flag'] = (
        (df['soc_in_pct'] > 100.0) | (df['soh_in_pct'] > 100.0) |
        (df['soc_out_pct'] > 100.0) | (df['soh_out_pct'] > 100.0)
    ).astype(int)

    expected_amount = df['list_price_inr'] - df['discount_inr']
    df['peak_offpeak_adjustment_inr'] = (df['amount_charged_inr'] - expected_amount).round(2)


    df['is_completed'] = (df['event_type'] == 'swap_completed').astype(int)
    df['is_failed'] = df['event_type'].isin(['failed_no_charged_battery', 'failed_system_error']).astype(int)
    df['is_abandoned'] = (df['event_type'] == 'abandoned_queue').astype(int)
    df['is_cancelled'] = (df['event_type'] == 'cancelled_by_rider').astype(int)


    df['is_valid_analysis_event'] = ((df['is_test_station'] == 0) & (df['is_near_duplicate'] == 0)).astype(int)

    if output_filepath:
        df.to_csv(output_filepath, index=False)
        print(f"Saved: '{output_filepath}' | Shape: {df.shape}")

    return df

df_swaps_clean = transform_swap_events('swap_events.csv')

Loading swap_events.csv (3.88M rows)...
Firmware-corrected rows: 139490
Flagging offline sync retries...
Near-duplicates flagged: 1600
Saved: 'swap_events_transformed.csv' | Shape: (3877013, 38)
